In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import datetime

In [8]:
# ==========================================
# 1. Load, Merge & Filter
# ==========================================
print("Loading Data...")
train = pd.read_csv('train.csv', parse_dates=['Date'], low_memory=False)
store = pd.read_csv('store.csv')

# Merge and filter: Open stores with Sales > 0
df = pd.merge(train, store, on='Store', how='left')
df = df[(df['Open'] == 1) & (df['Sales'] > 0)]

# Sort by Store/Date (Critical for Lag features)
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

Loading Data...


In [9]:
# ==========================================
# 2. Temporal Features
# ==========================================
print("Creating Temporal Features...")
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

# Duration since competition opened
df['CompetitionOpenSince'] = pd.to_datetime(dict(year=df.CompetitionOpenSinceYear.fillna(1900), 
                                                 month=df.CompetitionOpenSinceMonth.fillna(1), day=1))
df['CompetitionDaysOpen'] = (df['Date'] - df['CompetitionOpenSince']).dt.days.clip(lower=0)


Creating Temporal Features...


In [10]:
# ==========================================
# 3. Lags & Rolling Stats
# ==========================================
print("Creating Lags & Rolling Stats...")
grouped = df.groupby('Store')['Sales']

# Lags (1, 2, 7 days) - Shifted to avoid leakage
df['Sales_Lag1'] = grouped.shift(1)
df['Sales_Lag2'] = grouped.shift(2)
df['Sales_Lag7'] = grouped.shift(7)

# Rolling (7-day mean/std) - Shifted first
df['Sales_RollMean7'] = grouped.shift(1).rolling(7).mean()
df['Sales_RollStd7']  = grouped.shift(1).rolling(7).std()

# Drop initial NaNs caused by lags
df.dropna(subset=['Sales_Lag1', 'Sales_Lag7', 'Sales_RollMean7'], inplace=True)


Creating Lags & Rolling Stats...


In [11]:
# ==========================================
# 4. Fourier Terms (Seasonality)
# ==========================================
print("Adding Fourier Terms...")
# Weekly (period=7)
df['day_sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
df['day_cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7)

# Yearly (period=365)
day_of_year = df['Date'].dt.dayofyear
df['year_sin'] = np.sin(2 * np.pi * day_of_year / 365)
df['year_cos'] = np.cos(2 * np.pi * day_of_year / 365)


Adding Fourier Terms...


In [14]:
# ==========================================
# 5. Preprocessing
# ==========================================
print("Encoding & Imputing...")

# Fill NaNs (Fix: Use assignment instead of inplace=True to avoid FutureWarning)
df['CompetitionDistance'] = df['CompetitionDistance'].fillna(df['CompetitionDistance'].median())

# Fill remaining NaNs with 0
df.fillna(0, inplace=True)

# Label Encode Categoricals
categorical_cols = ['StateHoliday', 'StoreType', 'Assortment']
for col in categorical_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# Normalize Numerical Features [0, 1]
num_cols = ['CompetitionDistance', 'CompetitionDaysOpen', 'Sales_RollMean7', 'Sales_RollStd7']
df[num_cols] = MinMaxScaler().fit_transform(df[num_cols])


Encoding & Imputing...


In [15]:
# ==========================================
# 6. Time-Series Split
# ==========================================
print("Splitting Data (Train/Val/Test)...")

# Chronological Split: Test (Last 6 weeks), Validation (Previous 6 weeks)
max_date = df['Date'].max()
test_cut = max_date - pd.Timedelta(days=42)  # 6 weeks
val_cut  = test_cut - pd.Timedelta(days=42)  # 6 weeks

train_df = df[df['Date'] < val_cut]
val_df   = df[(df['Date'] >= val_cut) & (df['Date'] < test_cut)]
test_df  = df[df['Date'] >= test_cut]

# Define Features & Target
target = 'Sales'
ignore_cols = ['Date', 'Customers', 'Open', 'CompetitionOpenSince', 
               'CompetitionOpenSinceYear', 'CompetitionOpenSinceMonth']
features = [c for c in df.columns if c not in ignore_cols + [target]]

# Detailed Output for Verification
print("-" * 30)
print(f"Training Set:   {train_df.shape[0]} rows ({train_df.Date.min().date()} to {train_df.Date.max().date()})")
print(f"Validation Set: {val_df.shape[0]} rows ({val_df.Date.min().date()} to {val_df.Date.max().date()})")
print(f"Test Set:       {test_df.shape[0]} rows ({test_df.Date.min().date()} to {test_df.Date.max().date()})")
print("-" * 30)
print(f"Final Features ({len(features)}): \n{features}")

# Create Final X and y matrices
X_train, y_train = train_df[features], train_df[target]
X_val, y_val     = val_df[features], val_df[target]
X_test, y_test   = test_df[features], test_df[target]

print("\nPhase 2 Completed Successfully.")

Splitting Data (Train/Val/Test)...
------------------------------
Training Set:   745322 rows (2013-01-08 to 2015-04-23)
Validation Set: 36376 rows (2015-04-24 to 2015-06-04)
Test Set:       41415 rows (2015-06-05 to 2015-07-17)
------------------------------
Final Features (28): 
['Store', 'DayOfWeek', 'Promo', 'StateHoliday', 'SchoolHoliday', 'Id', 'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'Year', 'Month', 'Day', 'WeekOfYear', 'IsWeekend', 'CompetitionDaysOpen', 'Sales_Lag1', 'Sales_Lag2', 'Sales_Lag7', 'Sales_RollMean7', 'Sales_RollStd7', 'day_sin', 'day_cos', 'year_sin', 'year_cos']

Phase 2 Completed Successfully.
